### Imports

In [1]:
import pandas as pd
import os
from testgen.utils import *
from pathlib import Path
from dotenv import load_dotenv

In [2]:
base_path = Path().cwd()
load_dotenv(override=True)

True

In [3]:
# read data and rename columns
df = pd.read_excel(base_path / "data/sensor_requirements.xlsx")
df_examples = pd.read_excel(base_path / "data/sensor_examples.xlsx")

In [4]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Find Examples

In [5]:
def get_example_txt(df_examples, N_EXAMPLES=1):

    if N_EXAMPLES > 1:
        indexes_to_drop, examples = get_examples_from_df(
            df_examples.drop(columns=["selected"]), N_EXAMPLES
        )

    if N_EXAMPLES == 1:  # pick pre selected example when N_EXAMPLES = 1
        df_examples_n1 = df_examples[df_examples["selected"] == 1].copy()
        df_examples_n1.drop(columns=["selected"], inplace=True)
        indexes_to_drop, examples = get_examples_from_df(df_examples_n1, 1)

    # join all examples in a text format to add to prompt
    examples_txt = ""

    for e1 in examples.values():
        for e2 in e1:
            examples_txt += f"Requirement: {e2[0]}\n"
            # examples_txt += f"Vector: {e2[1]}\n"
            examples_txt += f"Target Sensor/s: {e2[1]}\n"
            examples_txt += "\n"

    return examples_txt, examples

In [7]:
examples_txt, examples = get_example_txt(df_examples, 1)

print("\n".join(examples_txt.split("\n")[-10:]))

Requirement: Drift control systems must adaptively use yaw rate feedback to optimize vehicle control during high-performance driving scenarios
Target Sensor/s: [yaw_rate]

Requirement: The power steering system must adapt the torque levels in response to detected road surface conditions (e.g., ice, water, gravel)
Target Sensor/s: [steering_torque]

Requirement: The control system must detect and mitigate power oversteer to maintain vehicle control by adjusting the wheel steering angle and acceleration pedal inputs
Target Sensor/s: [acceleration_pedal, wheel_steering_angle]




# Response Format

In [10]:
from testgen.prompts.Sensors import Sensors
from pydantic import Field, create_model

Sensors_t = Sensors.split("\n")


def clean_sensors_fn(x):
    x = x.split(":")
    x[0] = x[0][: x[0].find("(")]
    return x


Sensors_t = list(map(clean_sensors_fn, Sensors_t))

sensor_attrs = {}

for sensor in Sensors_t:
    sensor_attrs[sensor[0].strip().lower().replace(" ", "_")] = (
        int,
        Field(description=sensor[1].strip()),
    )


VectorFormat = create_model("VectorFormat", **sensor_attrs)

# LLM

In [11]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPrompt

In [ ]:
llm_models = {
    "ollama": ["phi4:latest"],
}

endpoint_attrs = {
    "ollama": {
        "api_key": os.getenv("OLLAMA_API_KEY"),
        "base_url": os.getenv("OLLAMA_ENDPOINT"),
    }
}

# Run for all Requirements

In [ ]:
for n_example in  [1, 3, 5, 8]:

    examples_txt, examples = get_example_txt(df_examples, n_example)

    for endpoint_name in llm_models.keys():

        for model_name in llm_models[endpoint_name]:

            print(f"Running {model_name} on {endpoint_name} examples {n_example}...")

            client = llm_client(endpoint_name, **endpoint_attrs[endpoint_name])

            results = client_invoke_sensor(
                endpoint_name,
                model_name,
                client,
                df,
                SystemPrompt,
                Sensors,
                examples_txt,
                UserPrompt,
                response_format=VectorFormat,
            )

            (
                number_of_requests,
                accuracy,
                avg_time_per_req,
                avg_token_per_req,
                avg_completion_token_per_req,
                total_tokens,
                total_completion_tokens,
                total_time,
            ) = calc_stats(results)

            results_file = save_responses(
                base_path,
                "sensor_single",
                model_name=model_name,
                n_examples=n_example,
                examples=examples,
                accuracy=accuracy,
                number_of_requests=number_of_requests,
                total_tokens=total_tokens,
                total_completion_tokens=total_completion_tokens,
                avg_token_per_req=avg_token_per_req,
                avg_completion_token_per_req=avg_completion_token_per_req,
                avg_time_per_req=avg_time_per_req,
                results=results,
            )

            print(f"Done")

Running gemma3:1b on ollama examples 1...


  0%|          | 0/97 [00:00<?, ?it/s]

Done
Running gemma3:1b on ollama examples 3...


  0%|          | 0/97 [00:00<?, ?it/s]

KeyboardInterrupt: 